[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fvalenzuelag/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/main/RA2/IL2.1/4-crewai-agent.ipynb)


# 4. Orquestación de Agentes con CrewAI

## Objetivos de Aprendizaje
- Entender el concepto de agentes colaborativos y los roles en un "equipo" (Crew).
- Aprender los componentes clave de CrewAI: `Agent`, `Task`, `Tool` y `Crew`.
- Definir un equipo de agentes (un investigador y un escritor) para realizar una tarea compleja.
- Ejecutar el `Crew` y observar cómo los agentes colaboran y se pasan el trabajo entre ellos.
- **Comprender las configuraciones específicas necesarias para usar CrewAI con GitHub Models API**.

## ¿Qué es CrewAI y por qué usarlo?

Mientras que LangChain proporciona los bloques de construcción fundamentales para crear un agente, **CrewAI** se especializa en la **orquestación de múltiples agentes autónomos**. La idea central es que, para resolver tareas complejas, es más eficiente tener un equipo de agentes especializados que colaboren, en lugar de un solo agente que intente hacerlo todo.

**Analogía:** Piensa en una agencia de marketing. No tienes una sola persona que es experta en investigación, redacción, diseño y redes sociales. Tienes un equipo donde cada miembro tiene un rol claro. CrewAI aplica este concepto a los agentes de IA.

**Ventajas de CrewAI:**
- **Roles Especializados**: Permite definir agentes con roles, objetivos (`goal`) e historias de fondo (`backstory`) específicas, lo que los hace más efectivos en su nicho.
- **Colaboración Autónoma**: Los agentes pueden delegar tareas entre ellos de forma autónoma.
- **Procesos Secuenciales y Jerárquicos**: Soporta flujos de trabajo donde las tareas se completan en un orden específico.
- **Claridad y Estructura**: El código es muy declarativo y fácil de leer, ya que se centra en definir el equipo y sus responsabilidades.

## Relación entre CrewAI y LangChain

CrewAI **utiliza LangChain internamente** para manejar los LLMs y las herramientas. Esto significa que:

1. **CrewAI** se encarga de la orquestación de agentes (el "director de orquesta")
2. **LangChain** proporciona la interfaz con los modelos y herramientas (los "instrumentos")

Esta relación requiere configuraciones específicas que veremos en este notebook.

### 1. Instalación y Configuración

### 2. Configuración Especial para GitHub Models API

**⚠️ IMPORTANTE:** Esta es la parte crítica que causa problemas si no se configura correctamente.

**El Problema:**
- CrewAI utiliza LangChain internamente
- LangChain busca las variables de entorno `OPENAI_API_KEY` y `OPENAI_API_BASE`
- Nosotros tenemos `GITHUB_TOKEN` y `OPENAI_BASE_URL`
- Necesitamos "mapear" nuestras variables a las que LangChain espera

**La Solución:**
Configuramos las variables de entorno que LangChain espera, usando nuestros valores de GitHub Models API.

In [1]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q crewai crewai-tools langchain-openai python-dotenv wikipedia


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.4/232.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 30.1 MB/s eta 0:00:00
   ━━━

In [2]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "GOOGLE_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
    os.environ.setdefault("LLM_MODEL", "openai/gpt-oss-120b")
    os.environ.setdefault("LLM_MODEL_SMALL", "openai/gpt-oss-120b")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

# Verificar que las credenciales llegaron
print("🔍 Verificando configuración:")
print(f"LLM_BASE_URL: {os.environ.get('LLM_BASE_URL', '❌ No configurado')}")
print(f"LLM_MODEL   : {os.environ.get('LLM_MODEL', '❌ No configurado')}")
print(f"LLM_API_KEY : {'✅ Configurado' if os.environ.get('LLM_API_KEY') else '❌ No configurado'}")

🔍 Verificando configuración:
LLM_BASE_URL: https://api.groq.com/openai/v1
LLM_MODEL   : openai/gpt-oss-120b
LLM_API_KEY : ✅ Configurado


### 3. Configuración del LLM

Ahora configuramos el LLM usando LangChain. Gracias a la configuración anterior, LangChain automáticamente usará nuestras variables de entorno mapeadas.

In [3]:
import wikipedia
from crewai import LLM

# Configurar el idioma de Wikipedia
wikipedia.set_lang('es')

# 🧠 Configurar el LLM para CrewAI
# CrewAI usa LiteLLM por debajo, no LangChain. El prefijo "openai/" le indica
# que hable el protocolo de OpenAI contra el `base_url` que le damos, lo que
# permite usar cualquier proveedor compatible (Groq, Mistral, etc.).
try:
    llm = LLM(
        model="openai/" + os.getenv("LLM_MODEL", "openai/gpt-oss-120b"),
        base_url=os.getenv("LLM_BASE_URL"),
        api_key=os.getenv("LLM_API_KEY"),
        temperature=0,
    )

    # Probar que el LLM funciona
    print("✅ LLM configurado exitosamente.")
    print(f"📝 Respuesta de prueba: {llm.call('Responde solo: funcionando')[:60]}")

except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    print("💡 Verifica que LLM_BASE_URL y LLM_API_KEY estén configurados correctamente.")
    llm = None

✅ LLM configurado exitosamente.
📝 Respuesta de prueba: funcionando


### 4. Definición de Herramientas con CrewAI

**⚠️ IMPORTANTE:** CrewAI requiere un enfoque específico para las herramientas.

**El Problema:**
- LangChain usa el decorador `@tool` para definir herramientas
- CrewAI requiere que las herramientas hereden de `BaseTool`
- Mezclar ambos enfoques causa errores

**La Solución:**
Usar `BaseTool` de `crewai_tools` para crear herramientas compatibles con CrewAI.

In [4]:
# En CrewAI 1.x, BaseTool vive en `crewai.tools` (antes estaba en `crewai_tools`)
from crewai.tools import BaseTool

# 🔧 Definimos la herramienta como subclase de BaseTool
class WikipediaSearchTool(BaseTool):
    name: str = "Wikipedia Search Tool"
    description: str = "Busca en Wikipedia un tema y devuelve un resumen detallado. Es ideal para obtener información sobre personas, lugares, conceptos históricos y científicos."

    def _run(self, query: str) -> str:
        """Ejecuta la búsqueda en Wikipedia"""
        try:
            # Configurar el idioma de Wikipedia
            wikipedia.set_lang("es")
            # Devolver un resumen detallado para que el escritor tenga más material
            return wikipedia.summary(query, sentences=5)
        except wikipedia.exceptions.PageError:
            return f"No se encontró ninguna página para '{query}'. Intenta con un término más específico."
        except wikipedia.exceptions.DisambiguationError as e:
            return f"La búsqueda para '{query}' es ambigua. Opciones disponibles: {e.options[:3]}. Especifica cuál te interesa."
        except Exception as e:
            return f"Error al buscar en Wikipedia: {str(e)}"

# Crear la instancia de la herramienta
wikipedia_tool = WikipediaSearchTool()
tools = [wikipedia_tool]

# Probar la herramienta
try:
    test_result = wikipedia_tool._run("Albert Einstein")
    print("✅ Herramienta de Wikipedia configurada y probada exitosamente.")
    print(f"📝 Resultado de prueba: {test_result[:100]}...")
except Exception as e:
    print(f"❌ Error probando la herramienta: {e}")

print(f"\n🔧 {len(tools)} herramienta(s) disponible(s) para los agentes.")

✅ Herramienta de Wikipedia configurada y probada exitosamente.
📝 Resultado de prueba: Albert Einstein (pronunciación en alemán: /ˈalbɛʁt ˈaɪnʃtaɪn/ ();​ Ulm, Reino de Wurtemberg, 14 de m...

🔧 1 herramienta(s) disponible(s) para los agentes.


### 5. Creación del Equipo de Agentes (Crew)

Ahora definimos nuestro equipo de agentes especializados:

1. **Investigador (Researcher)**: Busca información detallada usando Wikipedia
2. **Escritor (Writer)**: Transforma la información en una biografía bien redactada

**⚠️ IMPORTANTE:** Otro error común es el parámetro `verbose` en `Crew`.

In [7]:
from crewai import Agent, Task, Crew, Process

# 🕵️ Agente 1: El Investigador
researcher = Agent(
    role="Investigador Senior",
    goal="Encontrar información completa y precisa sobre personas históricas, científicos y figuras importantes utilizando fuentes confiables.",
    backstory="""Eres un investigador académico con años de experiencia en la búsqueda de información histórica y científica.
    Tu especialidad es encontrar datos precisos y relevantes en Wikipedia y otras fuentes confiables.
    Te enorgulleces de la exactitud de tu trabajo y siempre proporcionas contexto histórico relevante.
    No escribes biografías completas, tu trabajo es recopilar los datos más importantes y precisos.""",
    tools=tools,
    llm=llm,
    verbose=True,
    allow_delegation=False  # Este agente no delega trabajo
)

# ✍️ Agente 2: El Escritor
writer = Agent(
    role="Escritor de Biografías",
    goal="Crear biografías atractivas, bien estructuradas y fáciles de leer basadas en la información proporcionada por el investigador.",
    backstory="""Eres un escritor profesional especializado en biografías y divulgación científica.
    Tu habilidad única es transformar datos técnicos y históricos en narrativas cautivadoras que son
    tanto informativas como accesibles para el público general.
    Tienes un don especial para destacar los aspectos más interesantes de la vida de las personas
    y presentar sus logros de manera inspiradora.""",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("✅ Agentes creados exitosamente:")
print(f"🕵️ {researcher.role}")
print(f"✍️ {writer.role}")

✅ Agentes creados exitosamente:
🕵️ Investigador Senior
✍️ Escritor de Biografías


### 6. Definición de Tareas

Las tareas definen exactamente qué debe hacer cada agente y cómo se relacionan entre ellas.

In [9]:
# 🔍 Tarea 1: Investigación
research_task = Task(
    description="""Busca información detallada sobre Marie Curie en Wikipedia.
    Enfócate en:
    - Sus descubrimientos científicos más importantes
    - Su impacto en la ciencia y la sociedad
    - Datos biográficos clave (fechas, lugares, educación)
    - Sus premios y reconocimientos
    - Su legado científico

    Proporciona información precisa y bien organizada que el escritor pueda usar.""",
    expected_output="Un resumen detallado de 4-6 párrafos con los datos más importantes sobre la vida, descubrimientos y legado de Marie Curie.",
    agent=researcher
)

# ✍️ Tarea 2: Escritura
write_task = Task(
    description="""Usando la información recopilada por el investigador, escribe una biografía cautivadora de Marie Curie.

    Requisitos:
    - Mínimo 5 párrafos bien estructurados
    - Estilo atractivo y accesible
    - Incluir sus logros más importantes
    - Destacar su impacto en la ciencia
    - Formato Markdown con encabezados apropiados
    - Tono inspirador pero preciso

    La biografía debe ser educativa e inspiradora para lectores de todas las edades.""",
    expected_output="Una biografía completa en formato Markdown, bien estructurada y atractiva, de al menos 5 párrafos.",
    agent=writer,
    context=[research_task]  # Esta tarea depende del resultado de la investigación
)

print("✅ Tareas definidas exitosamente:")
print(f"🔍 Tarea de investigación: {research_task.description[:50]}...")
print(f"✍️ Tarea de escritura: {write_task.description[:50]}...")

✅ Tareas definidas exitosamente:
🔍 Tarea de investigación: Busca información detallada sobre Marie Curie en W...
✍️ Tarea de escritura: Usando la información recopilada por el investigad...


### 7. Ensamblaje del Equipo (Crew)

**⚠️ CONFIGURACIÓN CORREGIDA:** El parámetro `verbose` debe ser boolean, no entero.

In [10]:
# 🎯 CONFIGURACIÓN CORREGIDA: verbose debe ser boolean
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,  # Las tareas se ejecutan en orden
    verbose=True  # ✅ CORRECTO: boolean, no entero (verbose=2 causaría error)
)

print("✅ Equipo (Crew) ensamblado exitosamente:")
print(f"👥 {len(crew.agents)} agentes en el equipo")
print(f"📋 {len(crew.tasks)} tareas definidas")
print(f"🔄 Proceso: {crew.process}")
print(f"🔊 Verbose: {crew.verbose}")

✅ Equipo (Crew) ensamblado exitosamente:
👥 2 agentes en el equipo
📋 2 tareas definidas
🔄 Proceso: Process.sequential
🔊 Verbose: True


### 8. Ejecución del Crew

¡Ahora viene la magia! Ejecutamos el crew y observamos cómo los agentes colaboran.

In [11]:
# 🚀 Ejecutar el crew
if llm is None:
    print("❌ No se puede ejecutar el crew sin un LLM configurado.")
    print("💡 Verifica la configuración de las variables de entorno en las celdas anteriores.")
else:
    try:
        print("🚀 Iniciando ejecución del crew...")
        print("📝 Observa cómo los agentes colaboran paso a paso:\n")

        # En un notebook (Jupyter o Colab) ya hay un event loop corriendo, así que
        # `crew.kickoff()` falla con "invoked synchronously from within a running
        # event loop". La variante asíncrona es la que funciona aquí; en un script
        # .py normal se usa `crew.kickoff()` directamente.
        result = await crew.kickoff_async()

        print("\n" + "="*80)
        print("🏁 RESULTADO FINAL DEL CREW")
        print("="*80)
        print(result)

    except Exception as e:
        print(f"❌ Error durante la ejecución del crew: {e}")
        print("\n🔧 Posibles soluciones:")
        print("1. Verifica que LLM_API_KEY esté configurado correctamente")
        print("2. Asegúrate de que LLM_BASE_URL esté configurado")
        print("3. Confirma que LLM_MODEL exista en el proveedor configurado")
        print("4. Verifica que no hayas mezclado @tool con BaseTool")
        print("5. Asegúrate de que verbose=True (no verbose=2)")

        # Mostrar información de debugging
        print("\n🐛 Información de debugging:")
        import traceback
        traceback.print_exc()

🚀 Iniciando ejecución del crew...
📝 Observa cómo los agentes colaboran paso a paso:



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 43a48f37-7eba-4f67-8291-86704b1c4616                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Busca información detallada sobre Marie Curie en Wikipedia.                                              │
│      Enfócate en:                                                                                               │
│      - Sus descubrimientos científicos más importantes                                                          │
│      - Su impacto en la ciencia y la sociedad                                                                   │
│      - Datos biográficos clave (fechas, lugares, educación)                                                     │
│      - Sus premios y reconocimientos                                                                            │
│      - Su legado científico                                                                                     │
│                                                                                                                 │
│      Proporciona información precisa y bien organizada que el escritor pueda usar.                              │
│  ID: 6a3f8115-f399-4b9b-94ba-2b6decf0f5ea                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador Senior                                                                                     │
│                                                                                                                 │
│  Task: Busca información detallada sobre Marie Curie en Wikipedia.                                              │
│      Enfócate en:                                                                                               │
│      - Sus descubrimientos científicos más importantes                                                          │
│      - Su impacto en la ciencia y la sociedad                                                                   │
│      - Datos biográficos clave (fechas, lugares, educación)                                                     │
│      - Sus premios y reconocimientos                                                                            │
│      - Su legado científico                                                                                     │
│                                                                                                                 │
│      Proporciona información precisa y bien organizada que el escritor pueda usar.                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: wikipedia_search_tool                                                                                    │
│  Args: {'query': 'Marie Curie'}                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Escritor de Biografías                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Marie Curie: La Luz que Transformó la Ciencia                                                                │
│                                                                                                                 │
│  ## Una infancia de resistencia y sueños                                                                        │
│                                                                                                                 │
│  Nacida como **Maria Salomea Skłodowska** el 7 de noviembre de 1867 en Varsovia, entonces bajo el dominio del   │
│  Imperio ruso, Marie Curie creció en una familia de maestros comprometidos con la educación clandestina         │
│  polaca. Era la menor de cinco hijos y, a pesar de la prohibición rusa de la enseñanza superior a las mujeres,  │
│  asistió a la *universidad flotante* de Varsovia, una escuela clandestina que mantenía viva la llama del        │
│  saber. Esta temprana lucha contra la opresión sembró en ella una determinación férrea: la ciencia sería su     │
│  vía de escape y, al mismo tiempo, su herramienta para cambiar el mundo.                                        │
│                                                                                                                 │
│  ## París, la Sorbona y el encuentro con Pierre                                                                 │
│                                                                                                                 │
│  A los 24 años, impulsada por la sed de conocimiento, Marie se trasladó a París para reunirse con su hermana    │
│  Bronisława. Allí se matriculó en la Sorbona (Universidad de la Sèvres), donde obtuvo la licenciatura en        │
│  física en 1893 y, diez años después, el doctorado en ciencias. En 1895 contrajo matrimonio con el físico       │
│  **Pierre Curie**, y la pareja se convirtió rápidamente en una de las alianzas científicas más célebres de la   │
│  historia. Juntos, trabajaron en el pequeño laboratorio de Pierre, donde la curiosidad de Marie encontró su     │
│  primer gran desafío: la radiactividad recién descubierta por Henri Becquerel.                                  │
│                                                                                                                 │
│  ## Descubrimientos que cambiaron el mapa de la materia                                                         │
│                                                                                                                 │
│  A partir de 1896, Marie introdujo el término **radiactividad**, demostrando que no era una propiedad de los    │
│  compuestos sino del propio átomo. En 1898, ella y Pierre aislaron dos nuevos elementos radiactivos:            │
│  **polonio**, nombrado en honor a la Polonia natal de Marie, y **radio**. El aislamiento del radio puro,        │
│  completado en 1910, requirió la separación de más de **3 millones de veces** la masa del mineral de uranio     │
│  (pechblenda). Este esfuerzo titánico no solo reveló la naturaleza de los átomos, sino que abrió la puerta a    │
│  la química nuclear y a la física de partículas, sentando las bases de la energía atómica y la medicina         │
│  nuclear.                                                                                                       │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Usando la información recopilada por el investigador, escribe una biografía cautivadora de Marie Curie.  │
│                                                                                                                 │
│      Requisitos:                                                                                                │
│      - Mínimo 5 párrafos bien estructurados                                                                     │
│      - Estilo atractivo y accesible                                                                             │
│      - Incluir sus logros más importantes                                                                       │
│      - Destacar su impacto en la ciencia                                                                        │
│      - Formato Markdown con encabezados apropiados                                                              │
│      - Tono inspirador pero preciso                                                                             │
│                                                                                                                 │
│      La biografía debe ser educativa e inspiradora para lectores de todas las edades.                           │
│  Agent: Escritor de Biografías                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 43a48f37-7eba-4f67-8291-86704b1c4616                                                                       │
│  Final Output: # Marie Curie: La Luz que Transformó la Ciencia                                                  │
│                                                                                                                 │
│  ## Una infancia de resistencia y sueños                                                                        │
│                                                                                                                 │
│  Nacida como **Maria Salomea Skłodowska** el 7 de noviembre de 1867 en Varsovia, entonces bajo el dominio del   │
│  Imperio ruso, Marie Curie creció en una familia de maestros comprometidos con la educación clandestina         │
│  polaca. Era la menor de cinco hijos y, a pesar de la prohibición rusa de la enseñanza superior a las mujeres,  │
│  asistió a la *universidad flotante* de Varsovia, una escuela clandestina que mantenía viva la llama del        │
│  saber. Esta temprana lucha contra la opresión sembró en ella una determinación férrea: la ciencia sería su     │
│  vía de escape y, al mismo tiempo, su herramienta para cambiar el mundo.                                        │
│                                                                                                                 │
│  ## París, la Sorbona y el encuentro con Pierre                                                                 │
│                                                                                                                 │
│  A los 24 años, impulsada por la sed de conocimiento, Marie se trasladó a París para reunirse con su hermana    │
│  Bronisława. Allí se matriculó en la Sorbona (Universidad de la Sèvres), donde obtuvo la licenciatura en        │
│  física en 1893 y, diez años después, el doctorado en ciencias. En 1895 contrajo matrimonio con el físico       │
│  **Pierre Curie**, y la pareja se convirtió rápidamente en una de las alianzas científicas más célebres de la   │
│  historia. Juntos, trabajaron en el pequeño laboratorio de Pierre, donde la curiosidad de Marie encontró su     │
│  primer gran desafío: la radiactividad recién descubierta por Henri Becquerel.                                  │
│                                                                                                                 │
│  ## Descubrimientos que cambiaron el mapa de la materia                                                         │
│                                                                                                                 │
│  A partir de 1896, Marie introdujo el término **radiactividad**, demostrando que no era una propiedad de los    │
│  compuestos sino del propio átomo. En 1898, ella y Pierre aislaron dos nuevos elementos radiactivos:            │
│  **polonio**, nombrado en honor a la Polonia natal de Marie, y **radio**. El aislamiento del radio puro,        │
│  completado en 1910, requirió la separación de más de **3 millones de veces** la masa del mineral de uranio     │
│  (pechblenda). Este esfuerzo titánico no solo reveló la naturaleza de los átomos, sino que abrió la puerta a    │
│  la química nuclear y a la física de partículas, sentando las bases de la energía atómica y la medicina         │
│  nuclear.                                                                                                       │
│                                                       


🏁 RESULTADO FINAL DEL CREW
# Marie Curie: La Luz que Transformó la Ciencia  

## Una infancia de resistencia y sueños  

Nacida como **Maria Salomea Skłodowska** el 7 de noviembre de 1867 en Varsovia, entonces bajo el dominio del Imperio ruso, Marie Curie creció en una familia de maestros comprometidos con la educación clandestina polaca. Era la menor de cinco hijos y, a pesar de la prohibición rusa de la enseñanza superior a las mujeres, asistió a la *universidad flotante* de Varsovia, una escuela clandestina que mantenía viva la llama del saber. Esta temprana lucha contra la opresión sembró en ella una determinación férrea: la ciencia sería su vía de escape y, al mismo tiempo, su herramienta para cambiar el mundo.  

## París, la Sorbona y el encuentro con Pierre  

A los 24 años, impulsada por la sed de conocimiento, Marie se trasladó a París para reunirse con su hermana Bronisława. Allí se matriculó en la Sorbona (Universidad de la Sèvres), donde obtuvo la licenciatura en física e

## 🎓 Resumen de Configuraciones Críticas

### Problemas Comunes y Sus Soluciones

| **Problema** | **Síntoma** | **Solución** |
|-------------|-------------|-------------|
| **Variables de entorno** | `AuthenticationError: Incorrect API key` | Mapear `GITHUB_TOKEN` → `OPENAI_API_KEY` y `OPENAI_BASE_URL` → `OPENAI_API_BASE` |
| **Herramientas incompatibles** | `'Tool' object is not callable` | Usar `BaseTool` en lugar de `@tool` |
| **Parámetro verbose** | `ValidationError: Input should be a valid boolean` | Usar `verbose=True` en lugar de `verbose=2` |
| **LLM no configurado** | `llm is None` | Verificar que las variables de entorno estén configuradas |

### Configuración Correcta para GitHub Models API

```python
# 1. Mapear variables de entorno
os.environ["OPENAI_API_BASE"] = os.environ.get("OPENAI_BASE_URL", "")
os.environ["OPENAI_API_KEY"] = os.environ.get("GITHUB_TOKEN", "")

# 2. Configurar LLM sin parámetros explícitos
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 3. Usar BaseTool para herramientas
class MiTool(BaseTool):
    name: str = "Mi Herramienta"
    description: str = "Descripción de la herramienta"
    def _run(self, query: str) -> str:
        return "resultado"

# 4. Configurar Crew con verbose boolean
crew = Crew(
    agents=[agent1, agent2],
    tasks=[task1, task2],
    process=Process.sequential,
    verbose=True  # ✅ boolean, no entero
)
```

### Variables de Entorno Requeridas

```bash
export OPENAI_BASE_URL="https://models.inference.ai.azure.com"
export GITHUB_TOKEN="tu_token_de_github_aquí"
```

## 🚀 Conclusiones

### Lo que hemos aprendido:

1. **CrewAI vs LangChain**: CrewAI orquesta equipos de agentes, LangChain proporciona los componentes individuales.

2. **Configuración crítica**: Cuando usas CrewAI con GitHub Models API, necesitas mapear las variables de entorno correctamente.

3. **Herramientas compatibles**: CrewAI requiere `BaseTool`, no el decorador `@tool` de LangChain.

4. **Parámetros correctos**: `verbose` debe ser boolean, no entero.

5. **Colaboración de agentes**: Los agentes pueden trabajar en secuencia, pasándose información entre ellos.

### Próximos pasos:

En los siguientes módulos exploraremos:
- Sistemas de memoria para agentes
- Integración con herramientas externas
- Estrategias de planificación más avanzadas
- Observabilidad y monitoreo de agentes

### 💡 Tip para estudiantes:

**Siempre revisa las configuraciones de entorno cuando cambies entre frameworks.** Cada framework tiene sus propias convenciones y expectativas sobre cómo acceder a los servicios externos. La clave está en entender estas diferencias y configurar correctamente las interfaces entre ellos.